# 03. Revenue Exposure, Customer Lifetime & Risk Segmentation
**Project:** Telco Customer Churn & Retention Intelligence  
**Phase:** Phase 2 - Exploratory Data Analysis & Customer Behavior Analysis  
**Dataset:** `data/processed/telco_churn_cleaned.csv`

---

## Executive Overview & Business Framing
In telecommunications business intelligence, analyzing churn count in isolation can mislead executive leadership. High churn among low-paying accounts has minimal top-line impact, whereas moderate churn among high-tier enterprise or premium subscribers can devastate revenue.

In this notebook, we calculate:
1. **Customer Population Accounting:** Total, active, and churned subscriber headcounts.
2. **Monthly Recurring Revenue (MRR) Accounting:** Total MRR, active MRR, and churned MRR.
3. **Revenue at Risk:** The financial drag from churned accounts in this snapshot.
4. **Disproportionate Revenue Drag:** Why revenue loss outpaces account loss.
5. **Two-Dimensional Risk Segmentation Matrix:** Tenure Tiers × Spend Tiers.
6. **Actionable Retention Focus:** Pinpointing the highest ROI intervention targets.


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.autolayout'] = True

DATA_PATH = (Path.cwd().parent / "data/processed/telco_churn_cleaned.csv" 
             if Path.cwd().name == "notebooks" 
             else Path("data/processed/telco_churn_cleaned.csv")).resolve()
FIGURES_DIR = (Path.cwd().parent / "reports/figures" 
               if Path.cwd().name == "notebooks" 
               else Path("reports/figures")).resolve()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Data loaded successfully. Records: {len(df):,}")


Data loaded successfully. Records: 7,043


## 1. Population & Revenue Financial Accounting
We quantify top-line financial exposure across the total, retained, and churned customer base.


In [2]:
total_customers = len(df)
churned_customers = (df['Churn'] == 'Yes').sum()
active_customers = (df['Churn'] == 'No').sum()
overall_churn_rate = churned_customers / total_customers

total_mrr = df['MonthlyCharges'].sum()
churned_mrr = df[df['Churn'] == 'Yes']['MonthlyCharges'].sum()
active_mrr = df[df['Churn'] == 'No']['MonthlyCharges'].sum()
revenue_at_risk_pct = churned_mrr / total_mrr

avg_mrr_all = df['MonthlyCharges'].mean()
avg_mrr_churned = df[df['Churn'] == 'Yes']['MonthlyCharges'].mean()
avg_mrr_active = df[df['Churn'] == 'No']['MonthlyCharges'].mean()

print("=" * 60)
print("       EXECUTIVE FINANCIAL CHURN AUDIT")
print("=" * 60)
print(f"Total Customer Accounts     : {total_customers:,}")
print(f"Active (Retained) Accounts  : {active_customers:,} ({active_customers/total_customers*100:.2f}%)")
print(f"Churned Accounts            : {churned_customers:,} ({overall_churn_rate*100:.2f}%)")
print("-" * 60)
print(f"Total Monthly Recurring Rev : ${total_mrr:,.2f}")
print(f"Active Customer MRR         : ${active_mrr:,.2f} ({active_mrr/total_mrr*100:.2f}%)")
print(f"Churned MRR (Revenue at Risk): ${churned_mrr:,.2f} ({revenue_at_risk_pct*100:.2f}%)")
print("-" * 60)
print(f"Average Monthly Charge (All): ${avg_mrr_all:.2f}")
print(f"Average Monthly Charge (Act): ${avg_mrr_active:.2f}")
print(f"Average Monthly Charge (Chr): ${avg_mrr_churned:.2f}")
print("=" * 60)


       EXECUTIVE FINANCIAL CHURN AUDIT
Total Customer Accounts     : 7,043
Active (Retained) Accounts  : 5,174 (73.46%)
Churned Accounts            : 1,869 (26.54%)
------------------------------------------------------------
Total Monthly Recurring Rev : $456,116.60
Active Customer MRR         : $316,985.75 (69.50%)
Churned MRR (Revenue at Risk): $139,130.85 (30.50%)
------------------------------------------------------------
Average Monthly Charge (All): $64.76
Average Monthly Charge (Act): $61.27
Average Monthly Charge (Chr): $74.44


### Critical Financial Finding: Disproportionate Revenue Drag
- Churned accounts represent **26.54%** of the customer headcount.
- However, churned accounts represent **30.50%** of total Monthly Recurring Revenue ($139,130.85/month).
- This disparity occurs because churned customers pay an average of **$74.44/month**, which is **21.5% higher** than retained customers ($61.27/month).
- **Annualized Revenue Exposure (ARR):** The annualized recurring revenue at risk from this cohort represents **$1,669,570.20** per year.


In [3]:
# Visualizing Revenue Exposure Breakdown
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Headcount vs. Revenue Comparison
categories = ['Customer Accounts', 'Monthly Revenue (MRR)']
active_shares = [active_customers/total_customers*100, active_mrr/total_mrr*100]
churn_shares = [churned_customers/total_customers*100, churned_mrr/total_mrr*100]

x = np.arange(len(categories))
width = 0.35

axes[0].bar(x - width/2, active_shares, width, label='Retained / Active', color='#2b5c8f', edgecolor='black')
axes[0].bar(x + width/2, churn_shares, width, label='Churned (At Risk)', color='#d9534f', edgecolor='black')
axes[0].set_ylabel("Share of Total (%)", fontsize=11)
axes[0].set_title("Customer Volume Share vs. Revenue Share", fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories, fontsize=11)
axes[0].set_ylim(0, 85)
axes[0].legend(loc='upper right')

for i in range(len(categories)):
    axes[0].text(i - width/2, active_shares[i] + 1.5, f"{active_shares[i]:.1f}%", ha='center', fontweight='bold')
    axes[0].text(i + width/2, churn_shares[i] + 1.5, f"{churn_shares[i]:.1f}%", ha='center', fontweight='bold', color='#d9534f')

# Plot 2: Average Monthly Charge Comparison
bars = axes[1].bar(['Retained Customers', 'Churned Customers'], [avg_mrr_active, avg_mrr_churned],
                   color=['#2b5c8f', '#d9534f'], width=0.45, edgecolor='black')
axes[1].set_ylabel("Average Monthly Charges (USD)", fontsize=11)
axes[1].set_title("Average Monthly Charge: Retained vs. Churned", fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 95)
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 2, f"${h:.2f}", ha='center', fontweight='bold', fontsize=12)

fig1_path = FIGURES_DIR / "03_revenue_exposure_comparison.png"
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig1_path.resolve()}")


[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\03_revenue_exposure_comparison.png


## 2. Customer Segmentation: Tenure × Spend Matrix
To translate these findings into operational retention playbooks, we segment customers along two primary axes:
1. **Tenure Tiers:**
   - `New (0-12 months)`: High vulnerability onboarding window.
   - `Developing (13-36 months)`: Established accounts establishing loyalty.
   - `Mature (37-72 months)`: Long-term core customer base.
2. **Monthly Spend Tiers:**
   - `Low (<= $35/mo)`: Basic telephony or barebones DSL.
   - `Medium ($35 - $70/mo)`: Standard internet bundles.
   - `High (> $70/mo)`: Premium fiber optic, multi-line, and full media streaming tiers.


In [4]:
df['tenure_tier'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 36, 72],
    labels=['New (0-12m)', 'Developing (13-36m)', 'Mature (37-72m)']
)

df['spend_tier'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0, 35, 70, 150],
    labels=['Low (<=$35)', 'Medium ($35-$70)', 'High (>$70)']
)

segmentation_summary = df.groupby(['tenure_tier', 'spend_tier'], observed=False).agg(
    total_customers=('customerID', 'count'),
    churned_customers=('Churn', lambda s: (s == 'Yes').sum()),
    churn_rate_pct=('Churn', lambda s: (s == 'Yes').mean() * 100),
    total_mrr=('MonthlyCharges', 'sum'),
    churned_mrr=('MonthlyCharges', lambda s: s[df.loc[s.index, 'Churn'] == 'Yes'].sum())
).reset_index()

segmentation_summary['mrr_risk_share_pct'] = segmentation_summary['churned_mrr'] / total_mrr * 100
segmentation_summary = segmentation_summary.round(2)
print("--- Two-Dimensional Risk & Revenue Segmentation Matrix ---")
print(segmentation_summary.to_string(index=False))


--- Two-Dimensional Risk & Revenue Segmentation Matrix ---
        tenure_tier       spend_tier  total_customers  churned_customers  churn_rate_pct  total_mrr  churned_mrr  mrr_risk_share_pct
        New (0-12m)      Low (<=$35)              641                152           23.71   14067.35      3520.75                0.77
        New (0-12m) Medium ($35-$70)              670                291           43.43   35868.85     15562.20                3.41
        New (0-12m)      High (>$70)              875                594           67.89   72693.55     49871.30               10.93
Developing (13-36m)      Low (<=$35)              461                 24            5.21   10065.10       599.95                0.13
Developing (13-36m) Medium ($35-$70)              477                 78           16.35   26251.45      4273.10                0.94
Developing (13-36m)      High (>$70)              918                372           40.52   81072.10     33376.55                7.32
    Mature

In [5]:
# Heatmap Visualizations: Churn Rate (%) and Churned MRR ($) by Segment
pivot_churn_rate = segmentation_summary.pivot(index='tenure_tier', columns='spend_tier', values='churn_rate_pct')
pivot_churn_mrr = segmentation_summary.pivot(index='tenure_tier', columns='spend_tier', values='churned_mrr')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pivot_churn_rate, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Churn Rate (%)'}, linewidths=1)
axes[0].set_title("Churn Rate (%) across Tenure × Spend Segments", fontsize=13, fontweight='bold', pad=10)
axes[0].set_xlabel("Monthly Spend Tier", fontsize=11)
axes[0].set_ylabel("Account Tenure Tier", fontsize=11)

sns.heatmap(pivot_churn_mrr, annot=True, fmt=',.0f', cmap='Reds', ax=axes[1], cbar_kws={'label': 'Churned MRR (USD)'}, linewidths=1)
axes[1].set_title("Lost Monthly Revenue (MRR) by Segment", fontsize=13, fontweight='bold', pad=10)
axes[1].set_xlabel("Monthly Spend Tier", fontsize=11)
axes[1].set_ylabel("Account Tenure Tier", fontsize=11)

fig2_path = FIGURES_DIR / "03_segmentation_heatmaps.png"
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig2_path.resolve()}")


[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\03_segmentation_heatmaps.png


## 3. Executive Retention Priorities & Segment Playbook
Based on the financial cross-tabulation:

### Primary Vulnerability Segment: `New (0-12m)` × `High Spend (>$70)`
- **Subscriber Count:** 875 customers
- **Churn Rate:** **67.89%** (594 customers churned)
- **Monthly Revenue at Risk:** **$49,871.30/month** (representing **35.8% of all churned dollars** across the company!)
- **Strategic Imperative:** Premium subscribers experiencing buyer's remorse, onboarding friction, or technical setup issues in their first 90–180 days. Immediate white-glove onboarding and proactive tech support outreach required.

### Secondary Vulnerability Segment: `Developing (13-36m)` × `High Spend (>$70)`
- **Subscriber Count:** 918 customers
- **Churn Rate:** **40.52%** (372 customers churned)
- **Monthly Revenue at Risk:** **$33,376.55/month** (representing **24.0% of all churned dollars**)
- **Strategic Imperative:** Annual contract expiration cliff. Customers completing their first year seeking competitive renewal terms. Targeted 2-year contract renewal incentives with complimentary device upgrades.
